Configuracion del entorno

In [ ]:

!pip install -q torch transformers datasets tokenizers evaluate accelerate


import torch
import transformers
import datasets
import random 


if torch.cuda.is_available():
    # Crea un objeto 'device' que apunte a la GPU principal
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print(f" Éxito: GPU detectada y configurada: {gpu_name}")
    print(f" Versión de CUDA disponible para PyTorch: {torch.version.cuda}")
else:

    print(" No se detectó una GPU ")
    device = torch.device("none")



# Limpia la caché de la GPU 
# util en recompilacion, comentar si es primera vez

if device.type == 'cuda': 
    torch.cuda.empty_cache()
    print(" Memoria de la GPU liberada y lista para trabajar.")


Utilizamos CUDA, es lo mismo que ya se vio en clase
torch: Es la misma biblioteca que vimos  
transformers: Es la biblioteca de Hugging Face, tiene varias herramientas interesantes apra NLP de ahi vienen todas las siguientes herramientas 
datasets:para manejar conjuntos de datos
tokenizers:  para tokenizar texto 
evaluate: para evaluar modelos 
accelerate: para acelerar el entrenamiento de modelos mediante técnicas como mixed precision y distribución en múltiples GPUs

In [ ]:
import os
from datasets import Dataset


ruta_es = "./IALATIN/Raw_Data/CCMatrix.es-la.es"
ruta_la = "./IALATIN/Raw_Data/CCMatrix.es-la.la"
ruta_scores = "./IALATIN/Raw_Data/CCMatrix.es-la.scores"




if all(os.path.exists(ruta) for ruta in [ruta_es, ruta_la, ruta_scores]):

    try:

        with open(ruta_es, 'r', encoding='utf-8') as f_es, \
             open(ruta_la, 'r', encoding='utf-8') as f_la, \
             open(ruta_scores, 'r', encoding='utf-8') as f_scores:
            
            # .strip() elimina los saltos de línea (\n) al final de cada oración
            textos_es = [linea.strip() for linea in f_es]
            textos_la = [linea.strip() for linea in f_la]
            # Convertimos los scores a números flotantes
            scores = [float(linea.strip()) for linea in f_scores]

        #  importante confirmar que el paralelismo es perfecto
        if not (len(textos_es) == len(textos_la) == len(scores)):
            raise ValueError("Los archivos no tienen la misma cantidad de líneas. El corpus está desalineado.")

        #  Construir el objeto Dataset de Hugging Face
        # Esto agrupa nuestras listas independientes en el formato optimizado que necesitamos
        dataset = Dataset.from_dict({
            "es": textos_es,
            "la": textos_la,
            "score": scores
        })

        print(f"\nDataset construido exitosamente desde los archivos paralelos :D ")
        print(f" Total de pares de oraciones disponibles en memoria: {len(dataset):,}")

        # muestreo 
        print("\n 5 Pares de Oraciones Aleatorias")
        indices_aleatorios = random.sample(range(len(dataset)), 5)

        for i, idx in enumerate(indices_aleatorios):
            fila = dataset[idx]
            print(f"\n Ejemplo {i+1} (Índice {idx}):")
            print(f"    Latín:   {fila['la']}")
            print(f"    Español: {fila['es']}")
            print(f"    Score:   {fila['score']}")

    except Exception as e:
        print(f"\n error {e}")

else:
    print("la ruta esta incorrecta o faltan archivos")

 Leer los archivos línea por línea simultáneamente
Usamos encoding='utf-8' para evitar problemas con tildes o caracteres especiales
la libreria Dataset es de Hugging Face se usa para organizar los datos de manera eficiente y compatible con modelos de NLP ( Natural Language Processing)
en este caso en especifico agrupa las listas de textos y scores en un formato tabular


In [ ]:
import re

print("Iniciando el proceso de limpieza y filtrado del dataset...")

#  hiperparámetros de impieza 
MIN_SCORE = 1.05 

MIN_PALABRAS = 3
MAX_PALABRAS = 50

# Máxima desproporción permitida (ej. 2.0 significa que una oración 
# no puede tener más del doble de palabras que su contraparte)
MAX_PROPORCION = 2.0

def limpiar_par_oraciones(fila):
    # Regla 1: Filtro de puntuación de alineación
    if fila['score'] < MIN_SCORE:
        return False
        
    texto_la = fila['la']
    texto_es = fila['es']
    
    #  filtro de caracteres 
    if "http" in texto_la or "http" in texto_es or "<" in texto_la:
        return False
        
    # filtro de longitud 
    palabras_la = texto_la.split()
    palabras_es = texto_es.split()
    
    len_la = len(palabras_la)
    len_es = len(palabras_es)
    
    if len_la < MIN_PALABRAS or len_la > MAX_PALABRAS:
        return False
    if len_es < MIN_PALABRAS or len_es > MAX_PALABRAS:
        return False
        

    proporcion = max(len_la, len_es) / min(len_la, len_es)
    if proporcion > MAX_PROPORCION:
        return False
        
    return True

# filtro con multiprocesamiento 

num_nucleos = os.cpu_count() or 2
dataset_limpio = dataset.filter(limpiar_par_oraciones, num_proc=num_nucleos)

print(f" Pares originales: {len(dataset):,}")
print(f" Pares limpios retenidos: {len(dataset_limpio):,}")
print(f" Pares descartados: {len(dataset) - len(dataset_limpio):,}")


que significa es min score?
 Umbral de calidad del score (depende de cómo se generó CCMatrix, 
 valores superiores a 1.04 - 1.06 suelen indicar buena calidad en LASER

MAX_PROPORCION 
una regla de porque es importante usar esto es "GIGO" (basura entra basura sale) 
sin esto podria entrar una oracion en latin de 1 palabra y su contraparte 30, esto nos daria tremendas alucinadas
 2.0 significa que una oración 
 no puede tener más del doble de palabras que su contraparte)





In [ ]:
from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast
import os


print("Iniciando la configuración del Tokenizador BPE Compartido...")

tokenizador = ByteLevelBPETokenizer()

# generador para alimentar los datos

def iterador_de_textos(dataset, tamaño_lote=1000):
    for i in range(0, len(dataset), tamaño_lote):
        # Extraemos un lote y combinamos las listas de español y latín
        lote = dataset[i : i + tamaño_lote]
        yield lote["la"] + lote["es"]

# hiperparms del tokenizador
TAMAÑO_VOCABULARIO = 32000
FRECUENCIA_MINIMA = 2
TOKENS_ESPECIALES = [
    "<pad>", # Padding (relleno para igualar longitudes)
    "<bos>", # Beginning of sequence (inicio de oración)
    "<eos>", # End of sequence (fin de oración)
    "<unk>"  # Unknown (palabra desconocida)
]



#  tokenizador combinando ambos idiomas
tokenizador.train_from_iterator(
    iterador_de_textos(dataset_limpio),
    vocab_size=TAMAÑO_VOCABULARIO,
    min_frequency=FRECUENCIA_MINIMA,
    special_tokens=TOKENS_ESPECIALES
)

#  tokenizador en el entorno 
ruta_guardado = "./tokenizador_bpe_es_la"
if not os.path.exists(ruta_guardado):
    os.makedirs(ruta_guardado)

tokenizador.save_model(ruta_guardado)



print("Cargando el tokenizador entrenado y preparando las secuencias...")

tokenizador_hf = PreTrainedTokenizerFast(
    tokenizer_file="./tokenizador_bpe_es_la/tokenizer.json",
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>"
)

LONGITUD_MAXIMA = 50


def preparar_secuencias(lote):

    inputs = tokenizador_hf(
        lote["la"],
        max_length=LONGITUD_MAXIMA,
        padding="max_length", 
        truncation=True,      # Corta si la oración supera los 50 tokens
    )
    
    # tokenizamos el español 
    labels = tokenizador_hf(
        lote["es"],
        max_length=LONGITUD_MAXIMA,
        padding="max_length",
        truncation=True,
    )
    
    #  el modelo necesita los 'input_ids', la 'attention_mask' y los 'labels'
    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": labels["input_ids"]
    }

print("Aplicando la tokenización a todo el dataset limpio (esto puede tardar un poco)...")

# función paralelizada
num_nucleos = os.cpu_count() or 2
dataset_tokenizado = dataset_limpio.map(
    preparar_secuencias,
    batched=True,
    num_proc=num_nucleos,
    remove_columns=["la", "es", "score"]
)

#formato PyTorch
dataset_tokenizado.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


print(f"Dimensiones de input_ids (Latín): {dataset_tokenizado[0]['input_ids'].shape}")
print(f"Dimensiones de labels (Español): {dataset_tokenizado[0]['labels'].shape}")

 Es un método de tokenización que divide las palabras en subpalabras, lo que permite manejar palabras desconocidas y reducir el tamaño del vocabulario, pero no es a nivel de caracter, en algunos articulos Se encontro que a caracter en NMT alucina bastante, con respuestas hasta 6 veces mas largas https://aclanthology.org/D18-1461/
 
 Tambien en este bloque envolvemos el tokenizador BPE en la clase de Hugging Face, esto nos permite usar funciones avanzadas requeridas por PyTorch (como tensores dinámicos) [tensores dinamicos: permiten manejar secuencias de longitud variable sin necesidad de padding excesivo ]

 attention_mask: una matriz binaria (de unos y ceros) que le indica a la  red neuronal qué tokens son palabras reales (1) y cuáles son simple relleno <pad> (0), para que la red no pierda tiempo de cómputo 
 https://machinelearningmastery.com/a-gentle-introduction-to-attention-masking-in-transformer-models/ 
 



In [ ]:
from transformers import BartConfig, BartForConditionalGeneration

print("Configurando la arquitectura del modelo Transformer...")

#  Hiperparámetros (Ajustados para 8GB de VRAM)
configuracion = BartConfig(
    vocab_size=32000,           # Debe coincidir exactamente con el tokenizador de la Celda 4
    d_model=512,                # Dimensión de los vectores 
    encoder_layers=6,           # Número de capas del codificador (
    decoder_layers=4,           # Número de capas del decodificador
    encoder_attention_heads=8,  # Cabezales de atención paralela
    decoder_attention_heads=8,
    encoder_ffn_dim=1024,       # Tamaño de la red feed-forward interna
    decoder_ffn_dim=1024,
    max_position_embeddings=50, # Límite de la secuencia (coincide con la Celda anterior)
    pad_token_id=tokenizador_hf.pad_token_id,
    bos_token_id=tokenizador_hf.bos_token_id,
    eos_token_id=tokenizador_hf.eos_token_id,
    forced_eos_token_id=tokenizador_hf.eos_token_id,
)


#  NO un modelo preentrenado. Los pesos nacen aleatorios.
modelo = BartForConditionalGeneration(configuracion)


# Movemos físicamente las matrices de la RAM del sistema a la VRAM de la RTX 4060
modelo = modelo.to(device)



total_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Total de parámetros  {total_params:,}")

Bart ("Bidirectional and Auto-Regressive Transformers") fue desarrollado por Facebook AI Research (FAIR) y es un modelo de lenguaje basado en la arquitectura Transformer. Es un modelo de secuencia a secuencia (seq2seq) [a cada secuencia de entrada le corresponde una secuencia de salida] que es justo lo que necesitamos en un traductor.
Bart por si solo no es un modelo preentrenado, es una arquitectura, lo que significa que no tiene pesos preentrenados, sino que se inicializa con pesos aleatorios. Esto es importante porque nos permite entrenar el modelo desde cero con nuestro propio conjunto de datos, sin depender de conocimientos previos que podrían no ser relevantes para la tarea específica de traducción entre latín y español. 
https://quantumailabs.net/guide-to-bart-bidirectional-autoregressive-transformer/
https://www.digitalocean.com/community/tutorials/bart-model-for-text-summarization-part1  
https://github.com/NVIDIA/DeepLearningExamples/tree/master/PyTorch/LanguageModeling/BART 